In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import validation_metrics, inspect
print(validation_metrics.__file__)
print(inspect.getsource(validation_metrics.compute_normalised_wasserstein))

c:\Users\Sorana\Desktop\licenta\MY_PROJECT_PYOCIANINA\pyocyanin-ai\validation_metrics.py
from __future__ import annotations

import warnings
from typing import Optional, Sequence

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, ttest_ind
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_predict, train_test_split
from sklearn.preprocessing import StandardScaler

try:
    import ot  # Python Optimal Transport
    _OT_AVAILABLE = True
except ImportError:
    _OT_AVAILABLE = False
    warnings.warn("POT (ot) not installed - SWD metric will be skipped.", ImportWarning)

# Module-level constants (match notebook defaults)
FEATURE_KEYS    = ['Ip', 'Ep', 'FWHM', 'AUC', 'skewness', '

In [13]:
import pandas as pd
import numpy as np

from validation_metrics import ValidationGate, VoltammogramFidelityIndex

# Data
raw_potential_grid = 'raw/raw_potential_grid.csv'
raw_signals_real = 'raw/raw_signals_real.csv'
raw_signals_augmented = 'raw/raw_signals_augmented.csv'
# raw_signals_combined = 'raw/raw_combined_signals.csv'

gate = ValidationGate.from_csv(raw_potential_grid, raw_signals_real)

results = gate.evaluate_csv(raw_signals_augmented, run_tiers=[1, 2])

vfi = VoltammogramFidelityIndex.from_gate_results(results, verbose=True)

print(results['pff_df'].groupby('class_uM')['Wasserstein'].agg(['mean', 'median', 'count', 'max']))
print()
print(results['pff_df'].sort_values('Wasserstein', ascending=False).head(15))

print(results.get('delta_acf_peak', 'not in results dict'))
print([k for k in results.keys() if 'acf' in k.lower()])
print (results['pff_df']['Wasserstein'].describe())


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ✅ PASS              ║
║    KS mean frac                       : 0.962  (≥0.90)                ║
║    JSD mean / max                     : 0.0749 / 0.2259           ║
║    MMD²                               : 0.019505                ║
║    SWD mean / max                     : 0.1495 / 0.3119           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ✅ PASS              ║
║    PFF class pass frac                : 0.833  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9795  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.09473  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  NEW  VoltammogramFidelityIndex (VFI)  : 0.8669  [Good     ]  ║
║  NEW  P

In [9]:
import numpy as np
import pandas as pd
from validation_metrics import ValidationGate, assign_nearest_log_class

# --- 1. Load Data ---
raw_potential_grid = 'raw/raw_potential_grid.csv'
raw_signals_real = 'raw/raw_signals_real.csv'
raw_signals_augmented = 'raw/raw_signals_augmented.csv'
target_col = 'concentration'

E = pd.read_csv(raw_potential_grid).values.flatten()

# Real signals and labels
df_real = pd.read_csv(raw_signals_real)
y_real = df_real[target_col].values
X_real = df_real.drop(columns=[target_col]).values

# --- 2. Initialize Gate & Extract Features ---
gate = ValidationGate(E, X_real, y_real)

# ACCESS feat_real HERE:
feat_real = gate.feat_real  

# Evaluate synthetic data to get feat_synth
results = gate.evaluate_csv(raw_signals_augmented, target_col=target_col)
feat_synth = results['feat_synth']

df_aug = pd.read_csv(raw_signals_augmented)
y_synth = df_aug[target_col].values


# --- 3. Diagnostics ---

# How many synthetic samples per real class, after binning?
real_classes = np.unique(gate.y_real)
y_synth_binned = assign_nearest_log_class(y_synth, real_classes)
print("--- Synthetic Samples per Class ---")
print(pd.Series(y_synth_binned).value_counts().sort_index())

# Manually reproduce ONE cell -- Ep at 2.50 µM -- to see the raw numbers
r = feat_real[gate.y_real == 2.5]['Ep'].values
s = feat_synth[y_synth_binned == 2.5]['Ep'].values

print("\n--- Ep at 2.50 µM ---")
print('real Ep:', r)
print('synth Ep:', s)
print('real std:', r.std(ddof=1), 'synth std:', s.std(ddof=1))
print('pooled std used in normaliser:', np.std(np.concatenate([r, s]), ddof=1))


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ✅ PASS              ║
║    KS mean frac                       : 0.962  (≥0.90)                ║
║    JSD mean / max                     : 0.0749 / 0.2259           ║
║    MMD²                               : 0.019505                ║
║    SWD mean / max                     : 0.1495 / 0.3119           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ✅ PASS              ║
║    PFF class pass frac                : 0.833  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9795  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.09473  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  Tier 3 - Utility                     : ✅ PASS              ║
║    TSTR l

In [ ]:
##### big timegan test

import pandas as pd
import numpy as np

from validation_metrics import ValidationGate, VoltammogramFidelityIndex

# Data
raw_potential_grid = 'raw/raw_potential_grid.csv'
raw_signals_real = 'raw/raw_signals_real.csv'
raw_signals_augmented = 'gan_output/samples_trained_on_combined/timegan_signals.csv'
# raw_signals_combined = 'raw/raw_combined_signals.csv'

gate = ValidationGate.from_csv(raw_potential_grid, raw_signals_real)

results = gate.evaluate_csv(raw_signals_augmented, run_tiers=[1, 2,3,4])

vfi = VoltammogramFidelityIndex.from_gate_results(results, verbose=True)


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ❌ FAIL              ║
║    KS mean frac                       : 0.450  (≥0.90)                ║
║    JSD mean / max                     : 0.1088 / 0.5290           ║
║    MMD²                               : 0.132346                ║
║    SWD mean / max                     : 0.1489 / 0.4816           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ❌ FAIL              ║
║    PFF class pass frac                : 0.167  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9847  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.20179  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  Tier 3 - Utility                     : ❌ FAIL              ║
║    TSTR l

In [ ]:
##### big wgangp test

import pandas as pd
import numpy as np

from validation_metrics import ValidationGate, VoltammogramFidelityIndex

# Data
raw_potential_grid = 'raw/raw_potential_grid.csv'
raw_signals_real = 'raw/raw_signals_real.csv'
raw_signals_augmented = 'gan_output/samples_trained_on_combined/wgangp_signals.csv'
# raw_signals_combined = 'raw/raw_combined_signals.csv'

gate = ValidationGate.from_csv(raw_potential_grid, raw_signals_real)

results = gate.evaluate_csv(raw_signals_augmented, run_tiers=[1, 2,3,4])

vfi = VoltammogramFidelityIndex.from_gate_results(results, verbose=True)


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ❌ FAIL              ║
║    KS mean frac                       : 0.574  (≥0.90)                ║
║    JSD mean / max                     : 0.0719 / 0.2111           ║
║    MMD²                               : 0.039600                ║
║    SWD mean / max                     : 0.1439 / 0.4559           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ❌ FAIL              ║
║    PFF class pass frac                : 0.333  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9729  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.06285  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  Tier 3 - Utility                     : ❌ FAIL              ║
║    TSTR l

In [3]:

gate = ValidationGate.from_csv(raw_potential_grid, raw_signals_real)

results_sanity = gate.evaluate_csv(raw_signals_real, run_tiers=[1, 2])

vfi_perfect = VoltammogramFidelityIndex.from_gate_results(results_sanity, verbose=True)
print(f"VFI Sanity Check: {vfi_perfect:.4f}") 


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ✅ PASS              ║
║    KS mean frac                       : 1.000  (≥0.90)                ║
║    JSD mean / max                     : 0.0000 / 0.0000           ║
║    MMD²                               : -0.024235                ║
║    SWD mean / max                     : 0.0000 / 0.0000           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ✅ PASS              ║
║    PFF class pass frac                : 1.000  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9775  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.00000  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  NEW  VoltammogramFidelityIndex (VFI)  : 1.0000  [Excellent]  ║
║  NEW  

In [ ]:
import pandas as pd
import numpy as np
from validation_metrics import quick_compare

df_real = pd.read_csv('raw/raw_signals_real.csv')
E = pd.read_csv('raw/raw_potential_grid.csv').values.flatten()

y_real = df_real['concentration'].values
X_real = df_real.drop(columns=['concentration']).values

files_to_test = {
    'Augmented_Baseline': 'raw/raw_signals_augmented.csv',
    'TimeGAN':'gan_output/samples_trained_on_combined/timegan_signals.csv',
    'WGANGP':'gan_output/samples_trained_on_combined/wgangp_signals.csv'
}

batches = {}
for name, path in files_to_test.items():
    df_synth = pd.read_csv(path)
    y_synth = df_synth['concentration'].values
    X_synth = df_synth.drop(columns=['concentration']).values
    batches[name] = (X_synth, y_synth)


comparison_df = quick_compare(E, X_real, y_real, batches, run_tiers=[1, 2])


print(comparison_df.to_string())

                batch  KS_mean_frac  JSD_mean      MMD2  SWD_mean  PFF_class_frac  RS_R2_synth  delta_ACF     VFI VFI_label  PDF_overlap_mean  tier1_pass  tier2_pass
0  Augmented_Baseline        0.9624    0.0749  0.019505    0.1495           0.833       0.9795    0.09473  0.8669      Good            0.7072        True        True
1             TimeGAN        0.4503    0.1088  0.132346    0.1489           0.167       0.9847    0.20179  0.6351  Marginal            0.7204       False       False
2              WGANGP        0.5739    0.0719  0.039600    0.1439           0.333       0.9729    0.06285  0.7772      Good            0.7222       False       False


In [ ]:
import pandas as pd
import numpy as np
from validation_metrics import quick_compare

df_real = pd.read_csv('raw/raw_signals_real.csv')
E = pd.read_csv('raw/raw_potential_grid.csv').values.flatten()

y_real = df_real['concentration'].values
X_real = df_real.drop(columns=['concentration']).values

files_to_test = {
    'Augmented_Baseline': 'raw/raw_signals_augmented.csv',
    'TimeGAN':'gan_output/samples_trained_on_real/timegan_signals.csv',
    'WGANGP':'gan_output/samples_trained_on_real/wgangp_signals.csv'
}

batches = {}
for name, path in files_to_test.items():
    df_synth = pd.read_csv(path)
    y_synth = df_synth['concentration'].values
    X_synth = df_synth.drop(columns=['concentration']).values
    batches[name] = (X_synth, y_synth)


comparison_df = quick_compare(E, X_real, y_real, batches, run_tiers=[1, 2])


print(comparison_df.to_string())

                batch  KS_mean_frac  JSD_mean      MMD2  SWD_mean  PFF_class_frac  RS_R2_synth  delta_ACF     VFI VFI_label  PDF_overlap_mean  tier1_pass  tier2_pass
0  Augmented_Baseline        0.9624    0.0749  0.019505    0.1495           0.833       0.9795    0.09473  0.8669      Good            0.7072        True        True
1             TimeGAN        0.3118    0.1300  0.192612    0.2928           0.000       0.9523    0.51084  0.5166      Poor            0.6625       False       False
2              WGANGP        0.6882    0.0593  0.063509    0.1379           0.667       0.9538    0.01733  0.8849      Good            0.7610       False       False
